# 配套实践 14-01：训练动作条件动力学模型

本练习在二维状态系统中训练两个下一状态模型：一个只读取当前位置与速度，另一个同时读取动作并预测状态残差。数据中同一类状态会执行不同加速度，因此状态模型无法知道下一步向哪里变化。训练后还会比较分布内动作和更大未见动作的多步 rollout。依赖：PyTorch、NumPy、Matplotlib；CPU 即可运行。

<a href="https://qi-robotics.github.io/robot-world-model-tutorial/intermediate/14-action-conditioned-world-model/" target="_blank">在新标签页返回课程正文</a>

In [ ]:
import numpy as np  # 汇总训练曲线和多步 rollout 误差
import torch  # 生成动力学转移并训练两个预测网络
from torch import nn  # 使用多层感知机与回归损失
import matplotlib.pyplot as plt  # 绘制动作分叉、学习曲线和 rollout
torch.set_num_threads(2)  # 限制轻量实验的 CPU 线程开销
torch.manual_seed(141)  # 固定数据、模型初始化和批次顺序
np.random.seed(141)  # 固定 NumPy 侧随机过程
plt.rcParams["figure.dpi"] = 120  # 提高笔记本图像显示清晰度

## 1. 同一状态在不同动作下产生不同转移

状态由位置 p 和速度 v 组成，动作是加速度命令 a。真实系统还包含轻微位置相关力和速度阻尼。下面固定一个当前状态，只改变动作，直接观察下一状态的分叉。

In [ ]:
time_step = 0.12  # 设置离散动力学的控制时间间隔
def true_transition(states, actions):  # 定义用于产生训练标签的真实 toy 动力学
    positions = states[:, 0]  # 读取批量状态中的当前位置
    velocities = states[:, 1]  # 读取批量状态中的当前速度
    accelerations = actions[:, 0]  # 读取批量动作中的加速度命令
    nonlinear_force = -0.25 * torch.sin(positions)  # 加入随位置变化的轻微非线性恢复力
    next_velocities = 0.92 * velocities + time_step * (accelerations + nonlinear_force)  # 根据动作、阻尼和非线性力更新速度
    next_positions = positions + time_step * velocities + 0.5 * time_step ** 2 * accelerations  # 根据当前速度和动作更新位置
    return torch.stack([next_positions, next_velocities], dim=1)  # 返回下一位置与下一速度
fixed_state = torch.tensor([[0.35, -0.05]]).repeat(9, 1)  # 复制同一个当前状态供九种动作使用
shown_actions = torch.linspace(-1.0, 1.0, 9).unsqueeze(1)  # 建立从向左到向右的九个加速度
branched_next_states = true_transition(fixed_state, shown_actions)  # 计算同一状态下的九种真实未来
fig, axis = plt.subplots(figsize=(7.5, 4.3))  # 创建状态空间中的动作分叉图
axis.scatter(fixed_state[0, 0], fixed_state[0, 1], color="#172033", s=100, label="Current state")  # 标出共同当前状态
scatter = axis.scatter(branched_next_states[:, 0], branched_next_states[:, 1], c=shown_actions[:, 0], cmap="coolwarm", s=75, label="Next states")  # 用动作颜色显示不同下一状态
for next_state in branched_next_states:  # 逐个绘制从当前状态到下一状态的转移
    axis.plot([fixed_state[0, 0], next_state[0]], [fixed_state[0, 1], next_state[1]], color="#94a3b8", alpha=0.6)  # 显示动作导致的状态变化方向
fig.colorbar(scatter, ax=axis, label="Action acceleration")  # 添加动作大小颜色条
axis.set(title="One current state branches into different futures", xlabel="Position", ylabel="Velocity")  # 标注状态空间与图像含义
axis.legend()  # 显示当前状态和未来状态图例
axis.grid(alpha=0.2)  # 添加淡网格帮助观察分叉
fig.tight_layout()  # 调整图像边距
plt.show()  # 显示动作条件转移的必要性

**怎样理解结果：** 黑点是完全相同的当前状态，彩色点是不同加速度对应的下一状态。只读取当前状态的模型对九个样本只能给出同一个输出，最优 MSE 解接近这些未来的平均；读取动作的模型才能沿颜色变化预测正确分支。

## 2. 比较状态模型与动作条件残差模型

训练数据中的位置、速度和动作均随机采样。状态模型直接预测下一状态；动作条件模型预测下一状态减当前状态的残差。两个网络使用相同隐藏层规模。

In [ ]:
def make_transitions(sample_count, random_seed):  # 定义生成独立状态—动作—下一状态数据的函数
    generator = torch.Generator().manual_seed(random_seed)  # 为当前数据集建立独立随机生成器
    states = torch.empty(sample_count, 2).uniform_(-1.0, 1.0, generator=generator)  # 采样位置和速度状态
    actions = torch.empty(sample_count, 1).uniform_(-1.0, 1.0, generator=generator)  # 采样训练分布内加速度动作
    next_states = true_transition(states, actions)  # 使用真实动力学产生下一状态标签
    return states, actions, next_states  # 返回严格对齐的转移三元组
train_states, train_actions, train_next_states = make_transitions(5000, 21)  # 生成训练转移数据
test_states, test_actions, test_next_states = make_transitions(1200, 22)  # 生成独立未见测试转移
def build_mlp(input_dimension):  # 定义具有相同容量的预测网络构造函数
    return nn.Sequential(nn.Linear(input_dimension, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 2))  # 返回从输入到二维状态量的多层感知机
state_only_model = build_mlp(2)  # 创建忽略动作并直接预测下一状态的基线
action_residual_model = build_mlp(3)  # 创建读取状态和动作并预测状态残差的模型
state_optimizer = torch.optim.Adam(state_only_model.parameters(), lr=0.003)  # 为状态基线建立 Adam 优化器
action_optimizer = torch.optim.Adam(action_residual_model.parameters(), lr=0.003)  # 为动作条件模型建立 Adam 优化器
state_test_history = []  # 保存状态基线每轮测试误差
action_test_history = []  # 保存动作条件模型每轮测试误差
for epoch_index in range(45):  # 重复四十五轮小批量训练
    shuffled_indices = torch.randperm(len(train_states))  # 每轮打乱训练转移顺序
    for start_index in range(0, len(train_states), 250):  # 每次使用二百五十个转移更新模型
        batch_indices = shuffled_indices[start_index:start_index + 250]  # 取出当前批次索引
        state_predictions = state_only_model(train_states[batch_indices])  # 让基线只根据当前状态预测下一状态
        state_loss = ((state_predictions - train_next_states[batch_indices]) ** 2).mean()  # 计算状态基线的绝对预测 MSE
        state_optimizer.zero_grad()  # 清除状态基线上一批次梯度
        state_loss.backward()  # 反向传播状态基线误差
        state_optimizer.step()  # 更新状态基线参数
        action_inputs = torch.cat([train_states[batch_indices], train_actions[batch_indices]], dim=1)  # 拼接当前状态与对齐动作
        predicted_residuals = action_residual_model(action_inputs)  # 预测动作导致的状态变化量
        action_predictions = train_states[batch_indices] + predicted_residuals  # 把残差加回当前状态得到下一状态
        action_loss = ((action_predictions - train_next_states[batch_indices]) ** 2).mean()  # 计算动作条件下一状态 MSE
        action_optimizer.zero_grad()  # 清除动作模型上一批次梯度
        action_loss.backward()  # 反向传播动作条件预测误差
        action_optimizer.step()  # 更新动作条件模型参数
    with torch.no_grad():  # 关闭每轮测试过程的梯度记录
        state_test_loss = ((state_only_model(test_states) - test_next_states) ** 2).mean().item()  # 计算基线未见转移误差
        test_action_inputs = torch.cat([test_states, test_actions], dim=1)  # 组织测试状态与动作输入
        action_test_predictions = test_states + action_residual_model(test_action_inputs)  # 计算动作条件残差预测
        action_test_loss = ((action_test_predictions - test_next_states) ** 2).mean().item()  # 计算动作条件未见转移误差
    state_test_history.append(state_test_loss)  # 记录当前轮基线测试 MSE
    action_test_history.append(action_test_loss)  # 记录当前轮动作模型测试 MSE
fig, axis = plt.subplots(figsize=(8.8, 3.8))  # 创建两个 World Model 的测试误差曲线
axis.semilogy(np.arange(1, 46), state_test_history, color="#94a3b8", label="State only")  # 使用对数纵轴绘制忽略动作的误差
axis.semilogy(np.arange(1, 46), action_test_history, color="#2563eb", label="State + action residual")  # 绘制动作条件残差模型误差
axis.set(title="Action conditioning removes an irreducible prediction ambiguity", xlabel="Epoch", ylabel="Test MSE (log scale)")  # 标注模型差异和误差尺度
axis.legend()  # 显示两种模型图例
axis.grid(alpha=0.2)  # 添加淡网格帮助比较数量级
fig.tight_layout()  # 调整图像边距
plt.show()  # 显示动作条件对下一状态预测的影响

**怎样理解结果：** 状态基线很快到达无法继续下降的平台，因为它没有动作信息，只能平均随机动作造成的速度变化；动作条件残差模型的误差下降多个数量级，说明它学到了控制输入与状态变化的关系。这里的差异不是网络容量造成的，而是条件信息是否完整。

## 3. 单步准确不等于任意动作都可外推

我们从同一个初始状态滚动 45 步。第一组动作位于训练范围内；第二组把相同波形放大到约 1.6 倍，超出训练时的 [-1,1]。模型每一步都接收自己的预测状态。

In [ ]:
def rollout_models(action_sequence):  # 定义真实系统与两个 World Model 的多步 rollout
    true_state = torch.tensor([[0.2, 0.0]])  # 设置三个 rollout 共享的初始状态
    state_only_state = true_state.clone()  # 为状态基线复制初始状态
    action_model_state = true_state.clone()  # 为动作条件模型复制初始状态
    true_trace = [true_state.squeeze(0).numpy().copy()]  # 保存真实动力学轨迹
    state_only_trace = [state_only_state.squeeze(0).numpy().copy()]  # 保存忽略动作模型的轨迹
    action_model_trace = [action_model_state.squeeze(0).numpy().copy()]  # 保存动作条件模型的轨迹
    with torch.no_grad():  # 关闭 rollout 过程的梯度记录
        for action_value in action_sequence:  # 逐步执行给定候选动作序列
            action_tensor = torch.tensor([[action_value]], dtype=torch.float32)  # 把当前标量动作整理为批量张量
            true_state = true_transition(true_state, action_tensor)  # 使用真实动力学推进一步
            state_only_state = state_only_model(state_only_state)  # 让基线使用自己的预测继续滚动
            action_input = torch.cat([action_model_state, action_tensor], dim=1)  # 组织动作条件模型的当前预测状态与动作
            action_model_state = action_model_state + action_residual_model(action_input)  # 使用预测残差推进动作条件状态
            true_trace.append(true_state.squeeze(0).numpy().copy())  # 保存当前真实状态
            state_only_trace.append(state_only_state.squeeze(0).numpy().copy())  # 保存当前基线预测状态
            action_model_trace.append(action_model_state.squeeze(0).numpy().copy())  # 保存当前动作条件预测状态
    return np.stack(true_trace), np.stack(state_only_trace), np.stack(action_model_trace)  # 返回三条完整状态轨迹
rollout_steps = 45  # 设置多步预测长度
inside_actions = 0.85 * np.sin(np.linspace(0.0, 3.0 * np.pi, rollout_steps))  # 建立训练动作范围内的候选序列
outside_actions = 1.6 * np.sin(np.linspace(0.0, 3.0 * np.pi, rollout_steps))  # 建立明显超出训练范围的候选序列
inside_traces = rollout_models(inside_actions)  # 对分布内动作进行真实与模型 rollout
outside_traces = rollout_models(outside_actions)  # 对分布外动作进行真实与模型 rollout
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0), sharey=True)  # 创建分布内与分布外动作的轨迹对比
for axis, traces, title in zip(axes, [inside_traces, outside_traces], ["Actions inside training range", "Larger unseen actions"]):  # 依次绘制两种候选动作范围
    true_trace, state_trace, action_trace = traces  # 解包真实、状态基线和动作条件轨迹
    axis.plot(true_trace[:, 0], color="#172033", linewidth=2.5, label="True position")  # 绘制真实位置随 rollout 的变化
    axis.plot(state_trace[:, 0], color="#94a3b8", linestyle="--", label="State only")  # 绘制忽略动作模型的位置
    axis.plot(action_trace[:, 0], color="#2563eb", label="Action-conditioned")  # 绘制动作条件 World Model 的位置
    axis.set(title=title, xlabel="Rollout step", ylabel="Position")  # 标注动作范围与状态量
    axis.grid(alpha=0.2)  # 添加淡网格帮助观察误差累积
axes[0].legend()  # 在第一幅图显示三条轨迹图例
fig.tight_layout()  # 调整两幅 rollout 图间距
plt.show()  # 显示动作条件预测与外推风险

**怎样理解结果：** 在训练范围内，动作条件模型能够跟随真实位置变化，状态基线则无法响应动作波形。动作放大后，动作条件模型仍比基线合理，但误差明显增大并随 rollout 累积，因为这些控制输入未在训练中受到监督。

**本练习的结论：** 动作是 forward dynamics 的必要条件，但“包含动作”不代表模型可以可靠预测任意动作。规划前必须检查候选是否位于训练支持范围，并在长程 rollout 中估计模型误差与不确定性。